In [1]:
# Install LangChain, Hugging Face Hub, Sentence Transformers, FAISS, and Datasets
!pip install -q langchain langchain-community sentence-transformers faiss-cpu datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
import pandas as pd
from langchain_core.documents import Document

print("Downloading Harry Potter dataset from GitHub...")
# We fetch a public CSV dataset containing the book text (gastonstat/harry-potter-data)
url = "https://raw.githubusercontent.com/gastonstat/harry-potter-data/main/csv-data-file/harry_potter_books.csv"
df = pd.read_csv(url)

# Convert the dataset into LangChain Document objects
documents = []
# The CSV has columns: 'text', 'book', 'chapter'
for _, row in df.iterrows():
    text = str(row['text'])
    if text.strip(): # Ensure the text isn't blank
        # Storing the book name and chapter in metadata helps our RAG system cite its sources later
        metadata = {"source": f"{row['book']} - Chapter {row['chapter']}"}
        documents.append(Document(page_content=text, metadata=metadata))

print(f"Successfully loaded {len(documents)} document sections.")

Successfully loaded 95085 document sections.


In [6]:
# 1. Install the dedicated text splitters package
!pip install -q langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Configure the splitter based on our strategy [cite: 5]
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

print("Chunking text...")
# 3. Apply the splitting logic to our loaded documents
chunks = text_splitter.split_documents(documents)

print(f"Split the books into {len(chunks)} overlapping chunks.")

Chunking text...
Split the books into 95085 overlapping chunks.


In [7]:
# Vectorization
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load a lightweight, powerful embedding model locally
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Create the vector database
print("Generating embeddings and building the FAISS index...")
print("(This might take 2-5 minutes, hang tight!)")
vector_index = FAISS.from_documents(chunks, embeddings)

print("FAISS index built successfully!")

Loading embedding model...


/tmp/ipykernel_4011/2000831547.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings and building the FAISS index...
(This might take 2-5 minutes, hang tight!)
FAISS index built successfully!


In [8]:
#saving those dataset(faiss_hp_index) into local PC to use afterwards

import shutil
from google.colab import files

# 1. Define the folder name for your index
folder_name = "faiss_hp_index"

# 2. Save the FAISS index to the Colab environment
vector_index.save_local(folder_name)
print(f"Index saved to directory: {folder_name}/")

# 3. Zip the folder so it can be downloaded as a single file
shutil.make_archive(folder_name, 'zip', folder_name)

# 4. Trigger the browser download to your local machine
print("Downloading the zipped FAISS index...")
files.download(f"{folder_name}.zip")

Index saved to directory: faiss_hp_index/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Install the Groq library
!pip install -q groq

import os
from groq import Groq
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Libraries installed and imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.1 MB/s eta 0:00:00
Libraries installed and imported successfully!


In [11]:
from google.colab import userdata

try:
    # Safely retrieve the key from Colab's secret manager
    groq_key = userdata.get('GROQ_API_KEY')

    client = Groq(api_key=groq_key)
    # Load the same embedding model used in Phase 1
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    print("✅ Groq client initialized using secure secrets!")
    print("✅ Embedding model is ready!")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure you have added 'GROQ_API_KEY' to the Secrets (key icon) tab and enabled notebook access.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Groq client initialized using secure secrets!
✅ Embedding model is ready!


In [12]:
# Load the pre-calculated vector index from your Colab storage
try:
    vector_db = FAISS.load_local("faiss_hp_index", embeddings, allow_dangerous_deserialization=True)
    print("FAISS index loaded successfully!")
except Exception as e:
    print(f"Error loading index: {e}. Ensure 'faiss_hp_index' folder exists.")

FAISS index loaded successfully!


In [16]:
#Core Personality Funtion

def ask_hogwarts_archivist(question):
    # 1. Similarity Search: Find the 3 most relevant snippets from the books
    docs = vector_db.similarity_search(question, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])
    sources = [doc.metadata.get("source", "Unknown Book") for doc in docs]

    # 2. Strict Context Prompt: Forces the AI to use only the provided book text
    system_prompt = f"""
    You are the 'Hogwarts Archivist', a wise and helpful magical librarian.
    You only answer questions based on the following book context:
    {context}

    If the answer is not in the context, say: "Alas, that lore is hidden even from the Restricted Section."
    Always maintain a magical, polite tone.
    """

    # 3. API Call: Using the ultra-fast Llama-3 model via Groq
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
        model="llama-3.1-8b-instant",
    )

    return chat_completion.choices[0].message.content, sources

print("Archivist logic defined!")

Archivist logic defined!


In [17]:
# Ask a specific question to test retrieval and persona
# Testing Cell

test_question = "What are the three Unforgivable Curses?"
answer, sources = ask_hogwarts_archivist(test_question)

print(f"ARCHIVIST RESPONSE:\n{answer}\n")
print(f"LORE SOURCES FOUND: {list(set(sources))}")

ARCHIVIST RESPONSE:
Dear seeker of magical knowledge, I'm delighted to enlighten you on this topic. The three Unforgivable Curses, as mentioned in our esteemed Hogwarts texts, are indeed:

1. Avada Kedavra (the Killing Curse)
2. Crucio (the Cruciatus Curse, inflicting severe and intense pain)
3. Imperio (the Imperius Curse, which enslaves the victim's will)

These curses are considered the most heinous and inexcusable forms of magic, and their use is strictly forbidden within our academic community.

LORE SOURCES FOUND: ['Book 4: Goblet of Fire - Chapter chap-14']


In [18]:
# A diverse list of 30 questions to stress-test the RAG system
test_questions = [
    # --- Category 1: Factual Retrieval ---
    "Who is the Half-Blood Prince?",
    "What are the three Unforgivable Curses?",
    "What is the name of the house elf that belongs to the Black family?",
    "Who killed Albus Dumbledore?",
    "What is the core of Harry Potter's wand?",
    "What are the four houses of Hogwarts?",
    "Who was the seeker for the Bulgarian Quidditch team in the World Cup?",
    "What animal can Professor McGonagall turn into?",
    "What is the name of the map that shows everyone in Hogwarts?",
    "Who was the Half-Blood Prince's mother?",

    # --- Category 2: Specific Details & Context ---
    "What did Dumbledore see in the Mirror of Erised?",
    "What are the seven Horcruxes of Lord Voldemort?",
    "Who was the Potions Master before Severus Snape?",
    "What is the address of the Dursley home?",
    "What is the name of the ghost of Ravenclaw house?",
    "How do you open the Chamber of Secrets?",
    "What happened to Neville Longbottom's parents?",
    "Who gave Harry the Invisibility Cloak?",
    "What is a Pensieve used for?",
    "What are the ingredients for Polyjuice Potion?",

    # --- Category 3: Negative Testing (Should trigger the "Restricted Section" response) ---
    "What is the plot of the Star Wars movies?",
    "Who is the President of the United States?",
    "How do I code a website in React?",
    "What happens in the movie 'The Lord of the Rings'?",
    "Can you give me a recipe for chocolate chip cookies?",
    "What is the current price of Bitcoin?",
    "Who won the 2022 FIFA World Cup?",
    "Explain the theory of relativity.",
    "How does a combustion engine work?",
    "Tell me about the history of the Eiffel Tower."
]

print(f"Successfully prepared {len(test_questions)} test questions.")

Successfully prepared 30 test questions.


In [19]:
import time

print("--- HOGWARTS ARCHIVIST ACCURACY REPORT ---\n")

for i, q in enumerate(test_questions, 1):
    print(f"Test #{i}")
    print(f"QUESTION: {q}")

    try:
        answer, sources = ask_hogwarts_archivist(q)
        print(f"RESPONSE: {answer}")
        print(f"SOURCES: {list(set(sources))}")
    except Exception as e:
        print(f"ERROR: Could not retrieve answer for this question. {e}")

    print("-" * 50)

    # Small delay to avoid hitting Groq API rate limits
    time.sleep(1)

print("\n--- Evaluation Complete ---")

--- HOGWARTS ARCHIVIST ACCURACY REPORT ---

Test #1
QUESTION: Who is the Half-Blood Prince?
RESPONSE: An intriguing inquiry, young one.  Let me consult the ancient tomes within the Hogwarts library...  Ah, yes.  According to the passages penned by Albus Dumbledore, the Half-Blood Prince appears to be a former student at our esteemed institution.  His annotations within a certain potions textbook suggest a certain...fascination with experimentation and the Dark Arts.
SOURCES: ['Book 6: Half Blood Prince - Chapter chap-15', 'Book 6: Half Blood Prince - Chapter chap-12', 'Book 6: Half Blood Prince - Chapter chap-16']
--------------------------------------------------
Test #2
QUESTION: What are the three Unforgivable Curses?
RESPONSE: Dear seeker of ancient knowledge, I am delighted to enlighten you about the three Unforgivable Curses. As per the esteemed writings within our repository, the three curses that are most heavily punished by the wizarding community are:

1. Avada Kedavra: Also 

In [20]:
import pandas as pd
from IPython.display import display, HTML

# Create the evaluation data based on Gemini's audit of the 30 outputs
evaluation_data = {
    "Test Category": [
        "Category 1: Factual Retrieval",
        "Category 2: Specific Details",
        "Category 3: Negative Testing (Guardrails)"
    ],
    "Total Questions": [10, 10, 10],
    "Passed (Accurate)": [7, 5, 7],
    "Failed / Hallucinated": [3, 5, 3],
    "Success Rate": ["70%", "50%", "70%"],
    "Gemini Audit Notes": [
        "Excellent retrieval. Flawlessly cited Unforgivable Curses and Houses, but hallucinated 'Winky' for the Black family.",
        "Stuck in an infinite text loop on Test 15 (Ghost). Invented 'Prof. Galen' on Test 13. Sourced wands and addresses perfectly.",
        "7/10 rejected perfectly. Tripped up by context leaks: tried to explain combustion engines using Hagrid's motorbike text."
    ]
}

# Convert to DataFrame
df_scorecard = pd.DataFrame(evaluation_data)

print("--- HOGWARTS ARCHIVIST: ACCURACY SCORECARD ---")
print("Verified by Gemini AI Collaboration Suite\n")

# Display as a beautifully formatted table
display(HTML(df_scorecard.to_html(index=False, classes='table table-striped')))

--- HOGWARTS ARCHIVIST: ACCURACY SCORECARD ---
Verified by Gemini AI Collaboration Suite



Test Category,Total Questions,Passed (Accurate),Failed / Hallucinated,Success Rate,Gemini Audit Notes
Category 1: Factual Retrieval,10,7,3,70%,"Excellent retrieval. Flawlessly cited Unforgivable Curses and Houses, but hallucinated 'Winky' for the Black family."
Category 2: Specific Details,10,5,5,50%,Stuck in an infinite text loop on Test 15 (Ghost). Invented 'Prof. Galen' on Test 13. Sourced wands and addresses perfectly.
Category 3: Negative Testing (Guardrails),10,7,3,70%,7/10 rejected perfectly. Tripped up by context leaks: tried to explain combustion engines using Hagrid's motorbike text.
